[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/takeshun1984/NumeralAnalysisInGeophysics_SolidEarth/blob/main/06_FDM_2nd_homo_P-SV_Cerjan.ipynb)

前回のシミュレーションコードで計算領域端からの反射波が明瞭に確認できたため、以下のような吸収境界を計算領域端に設定する。
$$
G(p) = \exp{[-{\alpha(N_A-p)}^2]}
$$
というスポンジ型の境界条件を設定する。$\alpha$は減衰率で、$p$はモデル外周境界から数えたグリッド番号となる。左端（$i = 1,\cdots ,N_A$）では$p=i$で、右橋（$ i=NX-N_A+1,\cdots,NX$）となる。Cerjan et al. (1985)では、
$$
N_A=20 \\
\alpha=0.015
$$
が提案されています。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

NA = 20
ip = np.arange(0,30,1)
AL = 0.015

GC = []
for i in ip:
    if i < NA:
        G = np.exp(-(AL*(NA-i))**2)
    else:
        G = 1
        
    GC.append(G)

plt.figure(figsize=(6, 3))

# === プロット ===
plt.rcParams.update({'font.size': 10})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams["xtick.minor.visible"] = True  
plt.rcParams["ytick.minor.visible"] = True 

plt.plot(ip,GC)
plt.scatter(ip,GC)
plt.xlabel("Grid from model boundary")
plt.ylabel("Attenuation factor")

plt.show()

計算領域端で0.92と小さな減衰に見えるかもしれませんが、1ステップ進む毎に減衰されますし、あまり強い減衰構造の不連続を作ると吸収境界から反射波が発生してしまいます。

Cerjan et al. (1995)では$N_A=20$で固定となっていますが、少し拡張させて、
$$
G(p) = \exp{[-{\beta(1-\frac{p}{N_A})}^2]}
$$
と書き換えます。$\beta=0.09$とすると、$N_A=20$としたときに、Cerjan et al. (1995)と完全に一致します。
上記のアルゴリズムを、04_FDM_2nd_homo-P-SV.ipynbへ実装していきます。

In [ ]:
import os

# 領域設定など
SP = np.float32
NX, NZ = 400, 400
DX, DZ = 0.4, 0.4
DT = 0.02
NTMAX = 2000

# 震源情報
I0, K0 = NX // 2, NZ // 2
T0, TS = 5.0, 0.0
MXX, MZZ, MXZ, MO = 0.0, 0.0, 1.0, 1.0
DTXZ = DT / (DX * DZ)

# --- 震源情報 ---
I0, K0 = NX // 2, NZ // 2
T0, TS = 5.0, 0.0
MXX, MZZ, MXZ, MO = 0.0, 0.0, 1.0, 1.0

# --- 出力・減衰設定 ---
NTDEC, NXD, NZD = 50, 2, 2
ONAME0 = "output/psv.h."

# --- 関数定義 ---
def kupper(t, ts, tr):
    if ts <= t <= ts + tr:
        return 3 * np.pi * (np.sin(np.pi * (t - ts) / tr))**3 / (4 * tr)
    else:
        return 0.0

# 配列の初期化 0~NX+1
SXX = np.zeros((NX + 2, NZ + 2), dtype=SP)
SZZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
SXZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
VX  = np.zeros((NX + 2, NZ + 2), dtype=SP)
VZ  = np.zeros((NX + 2, NZ + 2), dtype=SP)

# 均質媒質
VP, VS, RO = 6.0, 3.5, 2.3
RIG_val = RO * VS**2
LAM_val = RO * VP**2 - 2.0 * RO * VS**2

# 各格子点に物性を割り当て
RIG = np.full((NX + 2, NZ + 2), RIG_val, dtype=SP)
LAM = np.full((NX + 2, NZ + 2), LAM_val, dtype=SP)
RHO = np.full((NX + 2, NZ + 2), RO, dtype=SP)

上記までは、前回のコードとの共通部分です。

In [ ]:
# --- 吸収境界条件 (Cerjan 1985) パラメータ ---
BETA, NA = 0.09, 20

# --- 吸収境界係数の計算 ---
gx1, gx2 = np.ones(NX+2, dtype=SP), np.ones(NX+2, dtype=SP)
gz1, gz2 = np.ones(NZ+2, dtype=SP), np.ones(NZ+2, dtype=SP)
for i in range(1, NA + 1):
    val1 = np.exp(-BETA*(1.0-(i-0.5)/NA)**2)
    val2 = np.exp(-BETA*(1.0-i/NA)**2)
    
    gx1[i], gz1[i], gx2[i], gz2[i] = val1, val1, val2, val2

    gx1[NX-i+1], gx2[NX-i+1], gz1[NZ-i+1], gz2[NZ-i+1] = val2, val1, val2, val1

GX1, GZ1 = np.meshgrid(gx1, gz1, indexing='ij')
GX2, GZ2 = np.meshgrid(gx2, gz2, indexing='ij')

上記のように、GX1、GZ1、GX2, GZ2として吸収境界を設定しました。各成分毎に2つ用意しているのは、セルの中央にある変数と端点にあたる変数で位置が半グリッドだけずれているためです。この措置をせずに実装しても大きな影響はありません。

以下の時間ループでは、
```
    SXX *= GX1 * GZ1
    SZZ *= GX1 * GZ1
    SXZ *= GX2 * GZ2
```
と
```
    VX *= GX2 * GZ1
    VZ *= GX1 * GZ2
```
の2箇所に端部分の減衰が乗じられています。

In [ ]:
# 出力ディレクトリの指定
os.makedirs("output", exist_ok=True)

# メインループ
print(f"{'Step':>5} / {NTMAX}: {'Time':>7} {'Vxmax':>12}")
for it in range(1, NTMAX + 1):
    T = it * DT

    # 1. 応力場の更新
    dxvx = (VX[1:NX+1, 1:NZ+1] - VX[0:NX, 1:NZ+1]) / DX
    dxvz = (VZ[2:NX+2, 1:NZ+1] - VZ[1:NX+1, 1:NZ+1]) / DX
    dzvx = (VX[1:NX+1, 2:NZ+2] - VX[1:NX+1, 1:NZ+1]) / DZ
    dzvz = (VZ[1:NX+1, 1:NZ+1] - VZ[1:NX+1, 0:NZ]) / DZ

    lam_v, rig_v = LAM[1:NX+1, 1:NZ+1], RIG[1:NX+1, 1:NZ+1]
    SXX[1:NX+1, 1:NZ+1] += ((lam_v + 2.0*rig_v) * dxvx + lam_v * dzvz) * DT
    SZZ[1:NX+1, 1:NZ+1] += ((lam_v + 2.0*rig_v) * dzvz + lam_v * dxvx) * DT
    SXZ[1:NX+1, 1:NZ+1] += rig_v * (dxvz + dzvx) * DT

    SXX *= GX1 * GZ1
    SZZ *= GX1 * GZ1
    SXZ *= GX2 * GZ2

    # 震源注入
    sdrop = MO * kupper(T, TS, T0) * DTXZ
    SXX[I0, K0] -= MXX * sdrop
    SZZ[I0, K0] -= MZZ * sdrop
    SXZ[I0-1:I0+1, K0-1:K0+1] -= MXZ * sdrop * 0.25

    # 2. 速度場の更新
    dxsxx = (SXX[2:NX+2, 1:NZ+1] - SXX[1:NX+1, 1:NZ+1]) / DX
    dxsxz = (SXZ[1:NX+1, 1:NZ+1] - SXZ[0:NX, 1:NZ+1]) / DX
    dzszz = (SZZ[1:NX+1, 2:NZ+2] - SZZ[1:NX+1, 1:NZ+1]) / DZ
    dzsxz = (SXZ[1:NX+1, 1:NZ+1] - SXZ[1:NX+1, 0:NZ]) / DZ

    VX[1:NX+1, 1:NZ+1] += (dxsxx + dzsxz) / RHO[1:NX+1, 1:NZ+1] * DT
    VZ[1:NX+1, 1:NZ+1] += (dxsxz + dzszz) / RHO[1:NX+1, 1:NZ+1] * DT
    
    VX *= GX2 * GZ1 
    VZ *= GX1 * GZ2 

    # 進捗表示とスナップショット出力
    if it % NTDEC == 0:
        vxmax = np.max(VX)
        print(f"{it:5d}/{NTMAX:5d}: T={T:6.2f}[s] vxmax={vxmax:.3e}")
        
        # 空間データ一括計算
        i_idx, k_idx = np.arange(1, NX+1, NXD), np.arange(1, NZ+1, NZD)
        ii, kk = np.meshgrid(i_idx, k_idx, indexing='ij')
        
        # 物理量の計算 (div, rot)
        d_vx_x = (VX[ii, kk] - VX[ii-1, kk]) / DX
        d_vz_z = (VZ[ii, kk] - VZ[ii, kk-1]) / DZ
        d_vx_z = (VX[ii, kk+1] - VX[ii, kk]) / DZ
        d_vz_x = (VZ[ii+1, kk] - VZ[ii, kk]) / DX
        
        out_data = np.column_stack([
            ((ii-1)*DX).ravel(), ((kk-1)*DZ).ravel(),
            VX[ii, kk].ravel(), VZ[ii, kk].ravel(),
            (d_vx_x + d_vz_z).ravel(), (d_vz_x - d_vx_z).ravel()
        ])
        np.savetxt(f"{ONAME0}{it:05d}.out", out_data, fmt='%9.3f %9.3f %12.3e %12.3e %12.3e %12.3e')


こちらの計算結果について、前回と同様に001000ステップ（20秒）の地震波動場を可視化して確認する。

In [ ]:
import pandas as pd
from PIL import Image

# --- 設定 ---
fact = 1e4 # scale factor
dt = 0.02
steps = range(50, 2001, 50)  # シミュレーションでは50ステップずつ出力
NX, NZ = 400 // 2, 400 // 2  # 格子数を空間的に間引いた

time_val = 30.0
step = int(time_val/DT)
step = int(step/100)*100
time_val = step*DT
filename = f"output/psv.h.{step:05d}.out"

# データ読み込み
out = pd.read_csv(filename, sep='\s+', names=('X','Z','VX','VZ','div','rot'))
    

fig, axs = plt.subplots(2, 2, figsize=(12, 10))
plt.rcParams.update({'font.size': 10})
    
# データ整形
X = out['X'].values.reshape(NZ, NX)-80
Z = out['Z'].values.reshape(NZ, NX)-80
VX = out['VX'].values.reshape(NZ, NX)
VZ = out['VZ'].values.reshape(NZ, NX)
VP = out['div'].values.reshape(NZ, NX)
VS = out['rot'].values.reshape(NZ, NX)
    
# 正規化
VX *= fact
VZ *= fact
VP *= fact*4
VS *= fact

# 左側：Vx
ax0 = axs[0,0]
im1 = ax0.pcolormesh(X, Z, VX, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_X$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_ylabel('Z from source [km]',fontsize=14)

# 右側：Vz
ax1 = axs[0,1]
im2 = ax1.pcolormesh(X, Z, VZ, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_Z$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
fig.colorbar(im2, ax=ax1, shrink=0.5).set_ticks([])

# 左側：div v
ax0 = axs[1,0]
im1 = ax0.pcolormesh(X, Z, VP, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{div}} \ \mathbf{{v}} \times 4$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_xlabel('X from source [km]',fontsize=14)
ax0.set_ylabel('Z from source [km]',fontsize=14)

# 右側：rot v
ax1 = axs[1,1]
im2 = ax1.pcolormesh(X, Z, VS, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{rot}} \ \mathbf{{v}}$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax1.set_xlabel('X from source [km]',fontsize=14)
fig.colorbar(im2, ax=ax1, shrink=0.5).set_ticks([])

for ax in axs.flatten():
    ax.set_xlim(-80, 80)
    ax.set_ylim(-80, 80)
    ax.set_aspect(1.0)
    ax.tick_params(direction="in", top=True, right=True, which='both')

plt.tight_layout()
plt.show()